# Worksheet 2 — MNIST Classification
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

Handwritten digit classification using the MNIST dataset (mnist_dataset.csv).

## Task 1: Load & Explore MNIST Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load MNIST from CSV
# mnist_dataset.csv: first column = label, remaining 784 = pixel values
import os
if os.path.exists('mnist_dataset.csv'):
    df = pd.read_csv('mnist_dataset.csv')
    y = df.iloc[:, 0].values
    X = df.iloc[:, 1:].values
else:
    # Fallback: use sklearn's built-in digits dataset
    from sklearn.datasets import load_digits
    digits = load_digits()
    X, y = digits.data, digits.target
    print('Note: Using sklearn digits dataset (mnist_dataset.csv not found)')

print(f'Dataset shape: {X.shape}')
print(f'Labels: {np.unique(y)}')
print(f'Class distribution: {np.bincount(y)}')

## Task 2: Visualise Samples

In [ ]:
# Show sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    idx = np.where(y == digit)[0][0]
    img_size = int(np.sqrt(X.shape[1]))
    axes[digit // 5, digit % 5].imshow(X[idx].reshape(img_size, img_size), cmap='gray')
    axes[digit // 5, digit % 5].set_title(f'Digit: {digit}')
    axes[digit // 5, digit % 5].axis('off')
plt.suptitle('MNIST Sample Digits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 3: Preprocessing & Train/Test Split

In [ ]:
# Normalise pixel values to [0, 1]
X_norm = X / 255.0 if X.max() > 1 else X

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')
print(f'Feature dimensions: {X_train.shape[1]}')

## Task 4: KNN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'KNN Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

## Task 5: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title(f'Confusion Matrix — KNN (Accuracy: {acc:.4f})', fontsize=13)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Task 6: SVM Classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.decomposition import PCA

# Reduce dimensions with PCA for faster SVM training
pca = PCA(n_components=50, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
print(f'Variance explained by 50 PCA components: {pca.explained_variance_ratio_.sum():.4f}')

svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train_pca, y_train)
svm_acc = accuracy_score(y_test, svm.predict(X_test_pca))
print(f'SVM Accuracy (PCA+RBF): {svm_acc:.4f} ({svm_acc*100:.2f}%)')

# Comparison
models = ['KNN (k=5)', 'SVM (RBF+PCA)']
accs = [acc, svm_acc]
plt.figure(figsize=(7, 4))
bars = plt.bar(models, accs, color=['#0EA5E9', '#6366F1'], width=0.4)
plt.ylim(0.9, 1.0)
plt.title('MNIST Classifier Comparison')
plt.ylabel('Accuracy')
for bar, a in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{a:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()